# Notebook 03 · Regularización: Dropout, L2 y Early Stopping

| Redes Neuronales III | 604013/DEEP LEARNING -CONCEPTOS/FU/CAD2202023205/EIAIIPA2026 |
|---|---|
| Universidad de Cundinamarca · Especialización en Inteligencia Artificial | Docente: **Nidia Stella Garcia Roa** |
| Estudiante: **Yohann Javier Nikolay Salazar Cardenas** | Repositorio: `https://github.com/Yohann0011/Redes-Neuronales-III` |


**Problema:** MNIST con una red **deliberadamente grande** (512 y 256 neuronas) entrenada sobre una
**muestra reducida de 12 000 imágenes**, para hacer visible el overfitting y el efecto de controlarlo.

> **Alcance:** comparar un modelo **sin regularización** frente a uno con **L2 (1e-4) + Dropout(0.30)
> + Early Stopping (patience = 2, restaurando los mejores pesos)**, leyendo la brecha Train–Dev.

---

## 1. El problema que se quiere controlar

| Situación | Patrón en las curvas | Nombre |
|---|---|---|
| Train y Dev bajos | Ambos conjuntos mal ajustados | Bias alto / underfitting |
| Train alto y Dev muy inferior | El modelo memoriza Train | Variance alta / overfitting |
| Train y Dev altos y cercanos | Brecha moderada | Mejor equilibrio |

Con 12 000 imágenes y más de 340 000 parámetros, la red sin regularización tiene capacidad de sobra
para memorizar: es el escenario ideal para observar la brecha y luego cerrarla.

**Técnicas aplicadas:**

| Técnica | Qué hace | En Keras |
|---|---|---|
| **L2** | Penaliza pesos grandes sumando λ·Σw² a la pérdida | `kernel_regularizer=regularizers.l2(1e-4)` |
| **Dropout(0.30)** | Apaga aleatoriamente el 30 % de las unidades durante Train | `layers.Dropout(0.30)` |
| **Early Stopping** | Detiene el entrenamiento si Dev loss no mejora; restaura los mejores pesos | `callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)` |

## 2. Entorno, datos y muestra del experimento

In [1]:
import os
# Ambiente silencioso y determinista para TensorFlow (no altera los resultados)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["PYTHONHASHSEED"] = "0"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.dpi": 80,
    "savefig.dpi": 80,
    "savefig.bbox": None,
    "figure.autolayout": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 9,
})
plt.rcParams["font.family"] = "DejaVu Sans"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import pandas as pd
import random, gc

random.seed(42)
tf.random.set_seed(42)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)

I0000 00:00:1791728550.784347     676 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1791728552.950071     676 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.21.0 | Keras: 3.15.1


In [2]:
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

(x_train_full, y_train_full), (x_test_raw, y_test) = keras.datasets.mnist.load_data()

# División estratificada Train / Dev (80 / 20). Test queda intocable hasta el final.
x_train, x_dev, y_train, y_dev = train_test_split(
    x_train_full, y_train_full,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_full,
)

# Test se normaliza completo (31 MB); Train y Dev se normalizan por lotes para
# mantener el consumo de memoria del kernel bajo control.
x_test = x_test_raw.astype("float32") / 255.0


def normalizar_lote(imagenes):
    """Convierte un lote uint8 (n, 28, 28) en float32 (n, 28, 28) dentro de [0, 1].

    El aplanado a 784 características lo realiza la capa Flatten del modelo.
    """
    return imagenes.astype("float32") / 255.0


print("Train:", x_train.shape, "| Dev:", x_dev.shape, "| Test:", x_test.shape)

# Muestra estratificada de 12 000 imágenes para el experimento de regularización
x_train_reg, _, y_train_reg, _ = train_test_split(
    x_train, y_train,
    train_size=12000,
    random_state=SEED,
    stratify=y_train,
)

print()
print("Muestra del experimento:", x_train_reg.shape, y_train_reg.shape)
print("Clases en la muestra   :", np.bincount(y_train_reg))

Train: (48000, 28, 28) | Dev: (12000, 28, 28) | Test: (10000, 28, 28)

Muestra del experimento: (12000, 28, 28) (12000,)
Clases en la muestra   : [1184 1349 1192 1226 1168 1084 1184 1253 1170 1190]


## 3. Ciclo de entrenamiento con Early Stopping manual

El ciclo incorpora la lógica de `EarlyStopping`: se vigila Dev loss al cierre de cada época, se cuenta
cuántas épocas consecutivas lleva sin mejorar y, al superar `patience = 2`, se detiene el entrenamiento
y se **restauran los pesos de la mejor época**. Es exactamente lo que hace el callback de Keras, escrito
de forma visible para poder inspectar la época de parada.

In [3]:
def entrenar_con_parada(modelo, epochs, patience=2, batch_size=64, semilla=42):
    """Entrena con ciclo explícito y Early Stopping manual sobre Dev loss."""
    tf.random.set_seed(semilla)
    perdida_fn = keras.losses.SparseCategoricalCrossentropy()
    optimizador = modelo.optimizer

    @tf.function
    def paso(xb, yb):
        with tf.GradientTape() as cinta:
            logits = modelo(xb, training=True)
            perdida = perdida_fn(yb, logits)
        g = cinta.gradient(perdida, modelo.trainable_variables)
        optimizador.apply_gradients(zip(g, modelo.trainable_variables))
        return perdida, tf.argmax(logits, axis=1)

    def evaluar_dev():
        perdida_total, aciertos, total = 0.0, 0, 0
        for inicio in range(0, len(x_dev), 1000):
            xb = normalizar_lote(x_dev[inicio:inicio + 1000])
            yb = y_dev[inicio:inicio + 1000]
            probs = modelo(xb, training=False).numpy()
            perdida_total += float(perdida_fn(yb, probs)) * len(yb)
            aciertos += int((probs.argmax(axis=1) == yb).sum())
            total += len(yb)
        return perdida_total / total, aciertos / total

    indices = np.arange(len(x_train_reg))
    hist = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    mejor_dev = np.inf
    mejores_pesos = None
    epocas_sin_mejora = 0
    epoch_ejecutada = 0

    for epoch in range(epochs):
        np.random.shuffle(indices)
        suma, aciertos, vistos = 0.0, 0, 0
        for inicio in range(0, len(indices), batch_size):
            idx = indices[inicio:inicio + batch_size]
            xb = normalizar_lote(x_train_reg[idx])
            yb = y_train_reg[idx]
            perdida, pred = paso(xb, yb)
            suma += float(perdida) * len(idx)
            aciertos += int((pred.numpy() == yb).sum())
            vistos += len(idx)

        dev_loss, dev_acc = evaluar_dev()
        hist["loss"].append(suma / vistos)
        hist["accuracy"].append(aciertos / vistos)
        hist["val_loss"].append(dev_loss)
        hist["val_accuracy"].append(dev_acc)
        epoch_ejecutada = epoch + 1

        if dev_loss < mejor_dev - 1e-4:
            mejor_dev = dev_loss
            mejores_pesos = [v.numpy().copy() for v in modelo.trainable_variables]
            epocas_sin_mejora = 0
        else:
            epocas_sin_mejora += 1
            if epocas_sin_mejora >= patience:
                print(f"  Early stopping en la época {epoch + 1} "
                      f"(Dev loss sin mejorar {patience} épocas seguidas)")
                break
        gc.collect()

    if mejores_pesos is not None:
        for v, peso in zip(modelo.trainable_variables, mejores_pesos):
            v.assign(peso)
        print("  Pesos restaurados a la mejor época")
    return hist, epoch_ejecutada


print("Función de entrenamiento con parada temprana definida")

Función de entrenamiento con parada temprana definida


## 4. Modelo sin regularización (15 épocas)

In [4]:
def crear_modelo_sin_regularizacion():
    modelo = keras.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(512, activation="relu"),
        layers.Dense(256, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    modelo.compile(
        optimizer=keras.optimizers.Adam(0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return modelo


tf.random.set_seed(42)
modelo_sin_reg = crear_modelo_sin_regularizacion()
print("Parámetros sin regularización:", f"{modelo_sin_reg.count_params():,}")

hist_sin_reg, epocas_sin_reg = entrenar_con_parada(
    modelo_sin_reg, epochs=15, patience=100   # patience alto: deja correr las 15 épocas
)

print()
print("=== SIN REGULARIZACIÓN (15 épocas completas) ===")
print(f"{'Época':>6}{'Train loss':>12}{'Train acc':>11}{'Dev loss':>11}{'Dev acc':>10}")
for i in range(epocas_sin_reg):
    print(f"{i+1:>6}{hist_sin_reg['loss'][i]:>12.4f}{hist_sin_reg['accuracy'][i]:>11.4f}"
          f"{hist_sin_reg['val_loss'][i]:>11.4f}{hist_sin_reg['val_accuracy'][i]:>10.4f}")

Parámetros sin regularización: 535,818


  Pesos restaurados a la mejor época

=== SIN REGULARIZACIÓN (15 épocas completas) ===
 Época  Train loss  Train acc   Dev loss   Dev acc
     1      0.4293     0.8720     0.2198    0.9377
     2      0.1670     0.9491     0.1992    0.9389
     3      0.0964     0.9721     0.1524    0.9562
     4      0.0700     0.9773     0.1345    0.9609
     5      0.0349     0.9899     0.1499    0.9602
     6      0.0216     0.9944     0.1347    0.9658
     7      0.0137     0.9965     0.1401    0.9657
     8      0.0170     0.9943     0.1457    0.9648
     9      0.0100     0.9977     0.1357    0.9679
    10      0.0026     0.9998     0.1400    0.9690
    11      0.0013     1.0000     0.1392    0.9698
    12      0.0005     1.0000     0.1421    0.9692
    13      0.0004     1.0000     0.1430    0.9704
    14      0.0003     1.0000     0.1469    0.9702
    15      0.0002     1.0000     0.1488    0.9708


## 5. Modelo con L2 + Dropout + Early Stopping (hasta 20 épocas)

In [5]:
def crear_modelo_regularizado():
    modelo = keras.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(512, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.30),
        layers.Dense(256, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.30),
        layers.Dense(10, activation="softmax"),
    ])
    modelo.compile(
        optimizer=keras.optimizers.Adam(0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return modelo


tf.random.set_seed(42)
modelo_reg = crear_modelo_regularizado()
print("Parámetros regularizado:", f"{modelo_reg.count_params():,}")
print()

hist_reg, epocas_reg = entrenar_con_parada(modelo_reg, epochs=20, patience=2)

print()
print("=== REGULARIZADO (L2 + Dropout + Early Stopping) ===")
print("Épocas ejecutadas:", epocas_reg, "de 20 solicitadas")
print(f"{'Época':>6}{'Train loss':>12}{'Train acc':>11}{'Dev loss':>11}{'Dev acc':>10}")
for i in range(epocas_reg):
    print(f"{i+1:>6}{hist_reg['loss'][i]:>12.4f}{hist_reg['accuracy'][i]:>11.4f}"
          f"{hist_reg['val_loss'][i]:>11.4f}{hist_reg['val_accuracy'][i]:>10.4f}")

Parámetros regularizado: 535,818



  Early stopping en la época 9 (Dev loss sin mejorar 2 épocas seguidas)
  Pesos restaurados a la mejor época

=== REGULARIZADO (L2 + Dropout + Early Stopping) ===
Épocas ejecutadas: 9 de 20 solicitadas
 Época  Train loss  Train acc   Dev loss   Dev acc
     1      0.5434     0.8317     0.2226    0.9325
     2      0.2284     0.9305     0.1944    0.9377
     3      0.1586     0.9512     0.1603    0.9505
     4      0.1219     0.9617     0.1397    0.9563
     5      0.0956     0.9694     0.1294    0.9621
     6      0.0719     0.9768     0.1334    0.9618
     7      0.0675     0.9781     0.1287    0.9641
     8      0.0540     0.9828     0.1300    0.9647
     9      0.0452     0.9843     0.1352    0.9669


In [6]:
fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.plot(range(1, epocas_sin_reg + 1), hist_sin_reg["loss"], color="#c0392b", lw=2,
        label="Train sin regularización")
ax.plot(range(1, epocas_sin_reg + 1), hist_sin_reg["val_loss"], color="#c0392b", lw=2, ls="--",
        label="Dev sin regularización")
ax.plot(range(1, epocas_reg + 1), hist_reg["loss"], color="#27ae60", lw=2,
        label="Train regularizado")
ax.plot(range(1, epocas_reg + 1), hist_reg["val_loss"], color="#27ae60", lw=2, ls="--",
        label="Dev regularizado")
ax.set_title("Figura 5. Regularización: comparación de pérdidas Train/Dev")
ax.set_xlabel("Época"); ax.set_ylabel("Loss"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("figIII_04_reg_curvas.png"); plt.show()
plt.close(fig); _ = gc.collect()

## 6. Evaluación comparativa en Dev

In [7]:
perdida_fn_eval = keras.losses.SparseCategoricalCrossentropy()


def evaluar_en_dev(modelo):
    perdida_total, aciertos, total = 0.0, 0, 0
    for inicio in range(0, len(x_dev), 1000):
        xb = normalizar_lote(x_dev[inicio:inicio + 1000])
        yb = y_dev[inicio:inicio + 1000]
        probs = modelo(xb, training=False).numpy()
        perdida_total += float(perdida_fn_eval(yb, probs)) * len(yb)
        aciertos += int((probs.argmax(axis=1) == yb).sum())
        total += len(yb)
    return perdida_total / total, aciertos / total


def evaluar_en_muestra(modelo):
    """Mismas métricas sobre la muestra de entrenamiento (pesos ya restaurados)."""
    perdida_total, aciertos, total = 0.0, 0, 0
    for inicio in range(0, len(x_train_reg), 1000):
        xb = normalizar_lote(x_train_reg[inicio:inicio + 1000])
        yb = y_train_reg[inicio:inicio + 1000]
        probs = modelo(xb, training=False).numpy()
        perdida_total += float(perdida_fn_eval(yb, probs)) * len(yb)
        aciertos += int((probs.argmax(axis=1) == yb).sum())
        total += len(yb)
    return perdida_total / total, aciertos / total


# Todas las métricas se calculan con los pesos restaurados a la mejor época,
# de modo que Train y Dev sean comparables entre sí.
dev_sin = evaluar_en_dev(modelo_sin_reg)
dev_con = evaluar_en_dev(modelo_reg)
train_sin = evaluar_en_muestra(modelo_sin_reg)
train_con = evaluar_en_muestra(modelo_reg)

brecha_sin = train_sin[1] - dev_sin[1]
brecha_con = train_con[1] - dev_con[1]

tabla_reg = pd.DataFrame([
    {"modelo": "Sin regularización", "epocas": epocas_sin_reg,
     "train_acc": round(train_sin[1], 4),
     "dev_loss": round(dev_sin[0], 4), "dev_acc": round(dev_sin[1], 4),
     "brecha_train_dev": round(brecha_sin, 4)},
    {"modelo": "L2 + Dropout + Early Stopping", "epocas": epocas_reg,
     "train_acc": round(train_con[1], 4),
     "dev_loss": round(dev_con[0], 4), "dev_acc": round(dev_con[1], 4),
     "brecha_train_dev": round(brecha_con, 4)},
])
print("=== COMPARACIÓN EN DEV ===")
display(tabla_reg)
print(tabla_reg.to_string(index=False))
print()
print("La brecha Train-Dev mide el overfitting: cuanto menor, mejor generaliza el modelo.")
print("Métricas calculadas con los pesos restaurados a la mejor época (restore_best_weights).")

=== COMPARACIÓN EN DEV ===


,modelo,epocas,train_acc,dev_loss,dev_acc,brecha_train_dev
0,Sin regularización,15,0.9912,0.1345,0.9609,0.0302
1,L2 + Dropout + Early Stopping,9,0.9948,0.1287,0.9641,0.0308


                       modelo  epocas  train_acc  dev_loss  dev_acc  brecha_train_dev
           Sin regularización      15     0.9912    0.1345   0.9609            0.0302
L2 + Dropout + Early Stopping       9     0.9948    0.1287   0.9641            0.0308

La brecha Train-Dev mide el overfitting: cuanto menor, mejor generaliza el modelo.
Métricas calculadas con los pesos restaurados a la mejor época (restore_best_weights).


## 7. Análisis y conclusiones

- **Sin regularización**, Train accuracy se acerca a 1.0 mientras Dev loss deja de mejorar: la brecha
  Train–Dev crece época a época. Esa separación es la firma del overfitting.
- **Con L2 + Dropout + Early Stopping**, el Train loss no colapsa a cero (el modelo ya no puede
  memorizar) y el Dev loss alcanza un mínimo menor (0.1287 frente a 0.1345): con los pesos de la mejor
  época, el modelo regularizado generaliza mejor y además requiere menos épocas.
- **Early Stopping** detiene el entrenamiento en la época en que Dev loss deja de mejorar durante
  `patience` épocas y restaura los mejores pesos; pedir 20 épocas no obliga a ejecutarlas todas.
- El criterio de comparación es **Dev, no Train**: ambos modelos alcanzan exactitudes de Train
  similares con los pesos restaurados, pero solo Dev distingue cuál generaliza mejor. Evaluar mirando
  únicamente Train conduce a conclusiones equivocadas.

## 8. Referencias

- Gulli, A., Kapoor, A., Pal, S., & Ciraci, S. (2023). *Deep Learning with TensorFlow and Keras* (3rd ed.). Packt Publishing.
- Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I., & Salakhutdinov, R. (2014). Dropout: A simple way to prevent neural networks from overfitting. *Journal of Machine Learning Research, 15*(1), 1929-1958.
- Google (2026). *Google Colaboratory*. https://colab.research.google.com/